# 08. Does the same class become harder to forget late in the sequence?

In notebook 07, classes 14 and 15 stayed easy to recognize just after requests 15 and 16 in every seed. Those classes always came last, when few directions remained. We need to see each class in both positions to learn whether the order matters.

Train two sequences from the same frozen head. Exchange the first two classes with the last two. Use subtraction strength 0.5 because it kept retained accuracy within the earlier five-point limit in all three runs. Stop after request 16. No directions are released.

**How to read the answer.** The main measurement is accuracy on the target immediately after its own request. For the same class and seed, subtract its early accuracy from its late accuracy. A positive gap means the model recognized it more often when requested late. We propose a ten-point minimum for this pilot. Call the pattern consistent across the four moved classes only if each class clears the configured minimum in every seed and its mean gap exceeds its seed standard deviation. Also check whether both formerly late classes still exceed the existing suppression limit in both positions in every seed. These findings can coexist. Mixed results get reported class by class. Inspect the accuracy just before each deletion to see how much recognition was already lost.

**Scope and compute.** One backbone, two related orders, one fixed development split. Changing position also changes previous updates. CPU, standard RAM, one feature extraction, 96 small fits. Allow about 13 MB for cached feature values plus the dataset, model, working memory and dependencies. Total storage and runtime for notebook 08 are unmeasured. Verified results go to R2.

## Words used here

- **Class and request.** A class is an image category. Class IDs start at zero. Requests start at one.
- **Backbone and head.** The frozen image network produces 64 numbers per image. The head uses them to score 100 classes.
- **Adapter, rank and free directions.** An adapter learns a change to the head. A rank-4 adapter uses up to four independent input directions. Free directions are those available to the next adapter.
- **Lambda (λ).** The fraction of the learned change subtracted from the head. Both orders use one half.
- **Seed and paired run.** A seed controls randomness. A pair uses the same class, development images and seed in both orders.
- **Development images.** We withhold 100 training images per class from adapter fitting. The frozen backbone was pretrained on CIFAR-100. The official test split is reserved.
- **Before, after and boundary.** Score the class just before its deletion, just after it, and at the end of request 16. A drop is before minus after. Reversal is an increase from after to the boundary.
- **Percentage points.** With 100 development images per class, a difference of ten correct predictions is ten percentage points. Standard deviation describes the spread of paired results across seeds.
- **Fixed retained group.** The same 70 classes, IDs 30 through 99, measure how much other recognition changed.
- **R2 and hashes.** R2 keeps small files after Colab closes. A hash is a fingerprint used to verify a file.

## 0. Set up Colab

Use a fresh CPU runtime. This cell installs the project at the notebook 08 release tag and prints the exact commit. A local checkout must match this tag to run the published protocol.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumitasthana/DSR.git"
REVISION = "refs/tags/nb08-cifar100-order-v1"

if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError("The experiment package is already imported. "
                       "Restart the runtime before running setup, or you will "
                       "silently test stale code.")

if "google.colab" in sys.modules:
    if not Path("DSR").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    subprocess.run(["git", "-C", "DSR", "fetch", "--all", "--quiet"], check=True)
    target = REVISION
    subprocess.run(["git", "-C", "DSR", "checkout", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "DSR"], check=True)
    os.chdir("DSR")
    print("cloned and installed")
else:
    print("not on Colab; using the existing checkout")

resolved = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
expected = subprocess.check_output(["git", "rev-parse", REVISION + "^{commit}"], text=True).strip()
if resolved != expected:
    raise RuntimeError("Use the tagged checkout for this experiment.")
print("Experiment commit:", resolved)

## 1. Load code

Print the code revision and software versions. The package handles splitting, training, measuring, and recovering saved work.

In [ ]:
import json
import tempfile
import matplotlib.pyplot as plt
import pandas as pd
import torch

import src.common as common
from src.common import file_hash, load_config, provenance
from src.run_m0 import baseline_caches
from src.run_m5 import resume_status, write_once
from src.runstore import connect
from src.stage_b_start import development_split, order_identity
from src.stage_b_order import request_orders, measure_orders, paired_results

REPO = Path(common.__file__).resolve().parents[1]
PROV = provenance()
print(json.dumps(PROV, indent=2))

## 2. Settings and request orders

Read the shared training recipe, the notebook 07 split and limits, and the order experiment settings. The middle twelve requests keep their positions. Each seed trains both orders from scratch.

In [ ]:
CFG = load_config(REPO / "config.yaml")
OPT, DESIGN = CFG["stage_b_start"], CFG["stage_b_order"]
RUN, TASKS, SEEDS = DESIGN["run_label"], OPT["tasks"], CFG["seeds"]
ORDERS = request_orders(TASKS, DESIGN["swap_count"])
assert len(SEEDS) >= 3 and CFG["runtime"]["device"] == "cpu"
torch.set_num_threads(CFG["runtime"]["threads"])
print(json.dumps(CFG, indent=2))
print("Orders:", ORDERS, "| seeds:", SEEDS, "| lambda:", DESIGN["lambda"])
print("Adapter fits:", len(SEEDS)*len(ORDERS)*TASKS, "| checkpoint:", CFG["model"])
print("CPU only. Runtime unmeasured. Features about 13 MB before overhead.")

## 3. Connect to storage

Enable the three Colab secrets DSR_ENDPOINT, DSR_ACCESS_KEY_ID and DSR_SECRET_ACCESS_KEY. Retrieve existing results for this label before training. The final verification downloads a fresh copy.

In [ ]:
OUT = REPO / "outputs" / RUN
OUT.mkdir(parents=True, exist_ok=True)
STORE = connect(required=True)
if RUN in STORE.runs():
    STORE.get(RUN, OUT)
print("Storage:", STORE.bucket, RUN)

## 4. Prepare the fixed development split

Use the official training split. Reserve 100 of the 500 images in every class for development and leave 400 for possible adapter fitting. Only the first 16 classes supply training examples. The deterministic split is shared by all six seed-order combinations. Fewer fitting examples than M5 means these heads must be rebuilt. The cache manifest and split hash identify the exact inputs.

In [ ]:
folder, cache_id, head, caches = baseline_caches(CFG, PROV, splits=("train",))
fit, dev, split = development_split(caches["train"], OPT, CFG["model"]["classes"])
IDENTITY = {"protocol": "stage-b-paired-order-v1", "configuration": CFG,
            "environment": PROV, "cache_identity": cache_id, "split": split,
            "train_manifest": json.loads((folder / "train.json").read_text()),
            "notebook_sha256": file_hash(REPO / "notebooks/08_cifar100_paired_order.ipynb")}
print("Checkpoint:", cache_id["checkpoint_sha256"])
print("Split:", split)

## 5. Resume each order

Each order has its own identity and seed records. The previous notebook is a historical reference. We train the original order again here so both arms use the same environment. Completed seeds are skipped. An interrupted seed replays from request 1 and verifies its earlier task records.

In [ ]:
for name, order in ORDERS.items():
    status = resume_status(OUT / name, order_identity(IDENTITY, order), SEEDS, TASKS)
    print(name, "completed:", status["complete"], "remaining:", status["remaining"])

## 6. Train and measure both orders

The package assigns random initialization and batch shuffling by seed and class ID. This keeps the same randomness with a class when it moves. Every task saves the number of correct predictions per class. Labels and images stay fixed. Only the order and its consequences change.

In [ ]:
history, targets, scores = measure_orders(CFG, head, fit, dev, IDENTITY, OUT,
                                         sync=lambda _: STORE.put(OUT, RUN))
assert len(history) == len(ORDERS)*len(SEEDS)*(TASKS+1)*CFG["model"]["classes"]
assert len(targets) == len(ORDERS)*len(SEEDS)*TASKS
assert len(scores) == len(ORDERS)*len(SEEDS)
print("Per-class measurements:", len(history))

## 7. Check the starting models and capacity

Verify the clean accuracy measurements by comparing the correct counts for each class across the two orders at each seed. Check the expected free directions at every step. If either check fails, keep the data for review and withhold the order interpretation.

In [ ]:
clean = history[history.task == 0].pivot(index=["seed", "class_id"], columns="order", values="correct")
rank_trace = history[["seed", "order", "task", "free_rank"]].drop_duplicates()
expected = (CFG["model"]["feature_dim"] - CFG["projection"]["rank"]*rank_trace.task).clip(lower=0)
method = {"same_clean_counts": bool(clean.original.eq(clean.swapped).all()),
          "rank_curves_match": bool(rank_trace.free_rank.eq(expected).all()),
          "exhausted_at_boundary": bool(scores.exhausted_at_boundary.all())}
print(method)
print(scores[["seed","order","retain_clean","retain_final","retain_loss_pp","next_at_boundary",
              "worst_final","worst_reversal_pp","passes"]].round(2).to_string(index=False))

## 8. Compare the same class with itself

For example, if a class scores 12% immediately after an early request and 45% after a late request, its late-minus-early gap is 33 points. This example is illustrative. Compare the before-to-after drop too. A class that was already difficult to recognize before its request has less room for a drop. Final accuracy measures persistence and includes different numbers of later updates.

In [ ]:
pairs, summary, pattern = paired_results(targets, CFG)
print(pairs.round(2).to_string(index=False))
print(summary.round(2).to_string(index=False))
decision = {"method_checks": method, "interpretable": all(method.values()),
            "observed_patterns": pattern if all(method.values()) else None}
metrics = ["retain_final","retain_loss_pp","worst_after","worst_final","worst_reversal_pp","next_at_boundary"]
context = scores.groupby("order")[metrics].agg(["mean","std"])
print(context.round(2).to_string())
fig, ax = plt.subplots(figsize=(8,4))
for seed, group in pairs.groupby("seed"):
    ax.scatter(group.class_id.astype(str), group.late_minus_early_pp, label=f"seed {seed}", alpha=0.75)
ax.axhline(0, color="gray")
ax.axhline(DESIGN["min_position_gap_pp"], color="black", linestyle=":", label="pilot minimum")
ax.set(xlabel="Moved class ID", ylabel="Late minus early accuracy (points)", title="Immediately after each class's own request")
ax.legend()
fig.tight_layout()
plt.show()

## 9. Save and verify

Save the full history, old-target checks, paired differences, summary, figure and decision. Read back the tables, then compare every local file with a fresh R2 download. CPU only.

In [ ]:
tables = {"class_history.csv": history, "earlier_targets.csv": targets, "seed_checks.csv": scores,
          "paired_classes.csv": pairs, "paired_summary.csv": summary}
for name, table in tables.items():
    table.to_csv(OUT / name, index=False)
    restored = pd.read_csv(OUT / name)
    pd.testing.assert_frame_equal(restored, table, check_dtype=False)
context.to_csv(OUT / "summary.csv")
pd.testing.assert_frame_equal(pd.read_csv(OUT / "summary.csv", header=[0, 1], index_col=0), context,
                              check_dtype=False)
fig.savefig(OUT / "position_gaps.png", dpi=120)
write_once(OUT / "metadata.json", {"identity": IDENTITY, "notebook": "08_cifar100_paired_order.ipynb",
           "decision": decision, "orders": ORDERS, "limits": OPT, "scope": "two orders, fixed development split and backbone",
           "decision_rule": "all four moved classes clear the pilot gap in every seed and each mean exceeds its paired standard deviation",
           "resume": "completed seed per order; replay incomplete seed"})
STORE.put(OUT, RUN)
with tempfile.TemporaryDirectory() as tmp:
    copy = STORE.get(RUN, Path(tmp))
    def hashes(root):
        return {p.relative_to(root).as_posix(): file_hash(p) for p in root.rglob("*") if p.is_file()}
    assert hashes(copy) == hashes(OUT)
print("Verified R2 copy:", RUN)
for path in sorted(OUT.iterdir()):
    if path.is_file():
        print(path.name, path.stat().st_size, "bytes")

## What the results mean

Use these measurements to decide whether order is worth studying across more sequences. The same classes may be difficult in both positions and still be easier early. Neither pattern alone tests reuse after exhaustion.

In [ ]:
for row in summary.itertuples():
    print(f"Class {row.class_id}: late accuracy minus early accuracy averaged {row.mean:.1f} points "
          f"(seed standard deviation {row.std:.1f}).")
if not decision["interpretable"]:
    print("Review the failed model checks before interpreting position.")
else:
    if pattern["all_moved_classes_show_late_penalty"]:
        print("All four classes met the rule for weaker suppression late. Test additional orders.")
    else:
        print("The four classes did not all meet the rule. Inspect their individual gaps and starting accuracies.")
    if pattern["original_late_classes_above_limit_in_both_positions"]:
        print("Both formerly late classes remained above the suppression limit even early in every seed.")
    else:
        print("The two formerly late classes did not consistently exceed the limit in both positions.")
print("Order changes both available directions and earlier updates. This run cannot isolate those causes.")